# LankaRain Ensemble Lab — ML & Pattern Recognition

**Task:** classify whether next-day rainfall is at least 1 mm in Sri Lankan city weather data, and compare Bagging, Boosting, Stacking, and a hybrid soft-voting ensemble.

**AI disclosure:** AI tools assisted with starter code and debugging. Review and understand all code and report measured results honestly.

## 1. Load and inspect data

Run this notebook from the extracted project folder so `ml_core.py` is importable. The source data downloads automatically if it is not already present.

In [ ]:
from ml_core import load_and_prepare_data, FEATURES, NUM_FEATURES, TARGET, chronological_split, build_models, evaluate_models, metrics_frame
import pandas as pd
import matplotlib.pyplot as plt

df = load_and_prepare_data()
print('Rows:', len(df))
print('Cities:', df['city'].nunique())
print('Date range:', df['date'].min(), 'to', df['date'].max())
df.head()

## 2. Explore target and dataset

The target is created from the next row within the same city: rain tomorrow = 1 when next day's precipitation is at least 1 mm. The features use today's values and earlier weather lags.

In [ ]:
df[TARGET].value_counts(normalize=True).rename('proportion')


In [ ]:
ax = df[TARGET].value_counts().sort_index().plot(kind='bar', title='Target distribution (rain tomorrow >= 1 mm)')
ax.set_xlabel('Rain tomorrow: 0 = no / light rain, 1 = >= 1 mm')
ax.set_ylabel('Records')
plt.show()

## 3. Chronological train/test split

The test set uses the latest 20% of target dates. This is more appropriate for time-dependent weather data than random shuffling.

In [ ]:
train, test, cutoff = chronological_split(df)
print('Target date test cutoff:', cutoff)
print('Train records:', len(train), 'Test records:', len(test))
print('Train target balance:', train[TARGET].value_counts(normalize=True).round(3).to_dict())
print('Test target balance:', test[TARGET].value_counts(normalize=True).round(3).to_dict())

## 4. Train Bagging, Boosting, Stacking, and Hybrid

- **Bagging:** Random Forest
- **Boosting:** Gradient Boosting
- **Stacking:** Random Forest + Gradient Boosting + Extra Trees base learners; Logistic Regression meta-model
- **Hybrid:** soft voting between Random Forest and Gradient Boosting

Training may take several minutes depending on the computer.

In [ ]:
results, fitted_models, train, test, cutoff = evaluate_models(df)
results_df = metrics_frame(results)
results_df.round(4)

## 5. Compare metrics

Use accuracy, balanced accuracy, precision, recall, F1, and ROC-AUC. If rain/no-rain classes are imbalanced, accuracy may hide poor performance on the minority class. Do not select the winner based on accuracy alone.

In [ ]:
results_df


In [ ]:
for metric in ['Accuracy', 'Balanced Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']:
    ax = results_df[metric].dropna().sort_values().plot(kind='barh', title=f'Model comparison — {metric}', figsize=(8,4))
    ax.set_xlim(0,1)
    ax.set_xlabel(metric)
    plt.tight_layout()
    plt.show()

## 6. Inspect Random Forest feature importance

This chart describes how the fitted Random Forest uses the input features. Importance is not causal evidence.

In [ ]:
rf = fitted_models['Bagging - Random Forest']
feature_names = rf.named_steps['prep'].get_feature_names_out()
importance = rf.named_steps['model'].feature_importances_
imp = pd.DataFrame({'feature': feature_names, 'importance': importance}).sort_values('importance', ascending=False).head(15)
imp

In [ ]:
ax = imp.sort_values('importance').plot(kind='barh', x='feature', y='importance', legend=False, figsize=(8,5), title='Top Random Forest feature importances')
plt.tight_layout()
plt.show()

## 7. Interpret your actual results (write this before submitting)

Fill these in using your run: **best model:** __; **best metric and score:** __; **model with highest recall:** __; **did the hybrid beat the single models?:** __.

Discuss why: Random Forest can reduce variance through averaging; Boosting learns sequential corrections; Stacking can learn complementary patterns but can also overfit or fail to improve if base models are similar; the hybrid soft vote can be robust but is not guaranteed to win. Mention rainfall class imbalance if visible.


## 8. Exported model files and dashboard

The `evaluate_models()` function saves `artifacts/models.joblib`, `artifacts/metrics.json`, and `artifacts/meta.json`. From Command Prompt in the project directory, start the UI using:

```bash
streamlit run app.py
```

The dashboard offers a city selector, weather sliders, model selector, prediction probability, metric comparison, rainfall history, and feature importance.
